<div dir="rtl">

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/hanenalmayouf/Computer_vision_Arabic-/blob/main/labs/04a_roi_evaluation.ipynb)


# 📈 أساسيات YOLO الجزء 4أ: التقييم المعتمد على العائد المادي (ROI) 💰

مرحباً بك في المحطة الأكثر أهمية لمديري المشاريع ومهندسي الحلول! 

في هذا المعمل، سننتقل من التفكير كمبرمجين (يهتمون بالدقة الرياضية فقط) إلى التفكير **كصناع قرار** (يهتمون بالربح والخسارة). سنكتشف لماذا قد نفضل أحياناً نموذجاً "أقل دقة" رياضياً ولكنه "أكثر توفيراً" مالياً.

--- 

## 🛠️ 1. تجهيز بيئة العمل

سنستخدم مكتبة `pandas` لتحليل البيانات وحساب التكاليف.

</div>

In [ ]:
# تثبيت المكتبات اللازمة
%pip install -q ultralytics pandas
import pandas as pd

<div dir="rtl">

## ⚖️ 2. فلسفة التقييم الموجه بالأعمال (ROI)

في الرياضيات، نعتبر أن كل خطأ هو "مجرد رقم". لكن في الحياة الحقيقية، الأخطاء لها أوزان مختلفة.

### 🚨 دراسة حالة: نظام اكتشاف السرقة في المتجر
تخيل أنك مهندس ذكاء اصطناعي لمتجر كبير. لديك نوعان من الأخطاء:

#### **1. الخطأ السلبي (False Negative - FN)**
*   **ماذا حدث؟** الكاميرا رأت سارقاً، لكن الموديل قال "هذا عميل عادي".
*   **النتيجة المادية:** خسارة قيمة المنتج (مثلاً 50 ريال).
*   **التأثير:** خسارة بسيطة ومتكررة.

#### **2. الخطأ الإيجابي (False Positive - FP)**
*   **ماذا حدث؟** العميل بريء، لكن الموديل صرخ "سارق!".
*   **النتيجة المادية:** قضية قانونية، خسارة سمعة المحل، خسارة العميل للأبد.
*   **التأثير:** خسارة ضخمة جداً (نقدرها بـ 2500 ريال لكل حالة).

--- 
### 💡 السؤال الجوهري:
أيهما أفضل؟ نموذج يمسك 100% من السارقين ولكنه يتهم 10 أشخاص بريئين يومياً؟ أم نموذج يمسك 80% من السارقين ولكنه لا يخطئ أبداً في حق الأبرياء؟

</div>

In [ ]:
# بيانات أداء النموذج عند مستويات ثقة مختلفة
# الـ threshold هو "حاجز الثقة"؛ كلما رفعناه، أصبح النموذج أكثر حذراً.
data = {
    'threshold': [0.1, 0.3, 0.5, 0.7, 0.8, 0.9], # مستوى الثقة المطلوبة
    'tp': [450, 400, 350, 250, 150, 50],       # True Positives (إمساك السارقين)
    'fp': [1000, 200, 50, 10, 2, 0],           # False Positives (اتهام الأبرياء)
    'fn': [50, 100, 150, 250, 350, 450]        # False Negatives (سرقات ضائعة)
}

df = pd.DataFrame(data)

# حساب مقاييس الأداء التقليدية
df['precision'] = df['tp'] / (df['tp'] + df['fp']) # الدقة
df['recall'] = df['tp'] / (df['tp'] + df['fn'])    # الاستدعاء
df['f1'] = 2 * (df['precision'] * df['recall']) / (df['precision'] + df['recall']) # التوازن الرياضي

print("📊 جدول الأداء التقني:")
print(df[['threshold', 'precision', 'recall', 'f1']])

<div dir="rtl">

## 💰 3. تحويل الأرقام إلى ريالات (Financial Transformation)

الآن سنقوم بعملية السحر التقني: سنضرب كل خطأ في قيمته المالية لنرى الحقيقة المرة.

### المعادلة المالية:
`التكلفة الكلية = (عدد السرقات الضائعة × 50) + (عدد الاتهامات الخاطئة × 2500)`

</div>

In [ ]:
COST_FN = 50   # خسارة 50 ريال لو ضاعت سرقة
COST_FP = 2500 # خسارة 2500 ريال لو اتهمنا بريء

def calculate_total_loss(row):
    # حساب التكلفة المادية بناءً على الأخطاء
    total_cost = (row['fn'] * COST_FN) + (row['fp'] * COST_FP)
    return total_cost

df['total_loss_sar'] = df.apply(calculate_total_loss, axis=1)

# تحديد أفضل عتبة لكل معيار
best_math_idx = df['f1'].idxmax()
best_money_idx = df['total_loss_sar'].idxmin()

print("==================================================")
print(f"🏆 الأفضل رياضياً (F1):  الثقة {df.loc[best_math_idx, 'threshold']}")
print(f"💰 الأفضل مالياً (ROI): الثقة {df.loc[best_money_idx, 'threshold']}")
print("==================================================")

print("\n💵 جدول الخسائر المالية لكل مستوى ثقة:")
print(df[['threshold', 'total_loss_sar', 'f1']])

<div dir="rtl">

## 🔬 4. تحليل النتائج: لماذا فاز الـ 0.8؟

لاحظ النتائج:
- عند **ثقة 0.5** (التوازن الرياضي): خسرنا مبالغ كبيرة بسبب وجود **50** اتهاماً خاطئاً (50 × 2500 = 125,000 ريال!).
- عند **ثقة 0.8**: ضاعت علينا سرقات أكثر، لكننا قللنا الاتهامات الخاطئة إلى **إثنين فقط**، مما وفر على الشركة مئات الآلاف.

### 💡 القاعدة العملية للنموذج:
> "إذا كان ثمن الغلطة باهظاً، ارفع عتبة الثقة (Threshold) ولا تجعل النموذج يتكلم إلا وهو متأكد جداً."

--- 

## 🏋️ تحدي إضافي: تشخيص طبي
تخيل أن النموذج الآن يكتشف **الأورام السرطانية**:
- **خطأ سلبي (FN)**: المريض مصاب والموديل قال سليم. (التكلفة: **حياة إنسان** - غالية جداً).
- **خطأ إيجابي (FP)**: المريض سليم والموديل قال مصاب. (التكلفة: فحص إضافي بـ 200 ريال).

**فكر:** هل سترفع عتبة الثقة هنا أم ستخفضها؟ (الجواب: ستخفضها جداً لأننا نريد مسك كل حالة حتى لو أخطأنا في السليمين).

--- 
## 🎉 نهاية المعمل
لقد أصبحت الآن مهندس ذكاء اصطناعي يمتلك رؤية تجارية ثاقبة. تذكر دائماً: الأرقام في الجدول هي وسيلة، والهدف الحقيقي هو حل مشكلة الواقع بأقل تكلفة ممكنة.

</div>